# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [4]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

ImportError: cannot import name 'from_json' from 'jiter' (unknown location)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [ ]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [ ]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [ ]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n1. Cat-Cow Stretch: Start on your hands and knees. Alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n\n2. Bird Dog: From hands and knees, extend the opposite arm and leg while keeping your core engaged. Hold each extension for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n3. Pelvic Tilts: Lie on your back with knees bent. Flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds. Repeat 8-12 times.\n\nThese exercises are gentle and targeted to help alleviate lower back discomfort. Always consult with a healthcare professional before starting any new exercise routine, especially if you have existing health issues.'

In [ ]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is essential for physical recovery, as the body repairs tissues and regenerates during sleep, particularly in the deep sleep stage. Sleep also plays a crucial role in mental well-being and cognitive function by consolidating memories and supporting learning. Adequate sleep (7-9 hours for adults) helps strengthen the immune system, reduce stress, and promote a healthy weight. Poor sleep or sleep disorders like insomnia can negatively affect health, increasing the risk for various conditions. Maintaining good sleep hygiene and creating an optimal sleep environment are important strategies to improve sleep quality and, consequently, overall health.'

In [ ]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water to stay hydrated and help alleviate dehydration-related headaches.\n- Applying cold or warm compresses to the head or neck for relief.\n- Resting in a dark, quiet room to help reduce headache symptoms.\n- Gentle massage of the temples and neck to relieve tension.\n- Using essential oils such as peppermint or lavender, which may help soothe headache pain.\n- Engaging in relaxation techniques like deep breathing, progressive muscle relaxation, or grounding exercises.\n- Practicing mindfulness or meditation to manage stress levels over the long term.\n- Maintaining a regular sleep schedule, using calming evening routines, and avoiding electronic screens before bed can also reduce overall stress and headache frequency.\n\nRemember, managing stress and headaches often involves both immediate relief techniques and long-term lifestyle adjustments.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [ ]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [ ]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [ ]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back up (cat) and letting it sag down (cow). Repeat 10-15 times.\n- **Bird Dog:** From your hands and knees, extend your opposite arm and leg while keeping your core engaged. Hold each for 5 seconds, then switch sides. Aim for 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and may prevent future issues.'

In [ ]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a crucial role in overall health. It helps the body repair and regenerate during the deep sleep stages, especially in Stage 3, which is essential for physical restoration. Adequate sleep also supports brain functions such as memory and learning, particularly during REM sleep when the brain is active and dreams occur. Maintaining a regular sleep schedule and creating an optimal sleep environment—such as a cool, dark, quiet, and comfortable surroundings—can improve sleep quality. Consistently getting 7-9 hours of sleep per night is vital for overall wellness, impacting immunity, mental health, and physical health.'

In [ ]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include relaxation techniques such as progressive muscle relaxation, meditation, and deep breathing exercises. Herbal teas like chamomile or valerian root may also help reduce stress and promote relaxation. Additionally, staying well-hydrated, ensuring adequate sleep, and managing lifestyle triggers can be beneficial in alleviating headaches related to stress.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

The Example
Query: "What is the recommended daily intake of Vitamin B12?"
Why BM25 wins here:
BM25 will exact-match on "Vitamin B12" as a specific token sequence. Dense embeddings encode "Vitamin B12" into a region of semantic space that's close to "vitamins," "supplements," "nutrition," "B-complex," "micronutrients" — all of which are semantically related but not the same thing.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [ ]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [ ]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [ ]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include gentle stretching and strengthening activities such as:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back up (like a cat) and letting it sag down (like a cow). Aim for 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each for about 5 seconds, then switch sides. Do 10 repetitions on each side.\n- **Pelvic Tilts:** Lie on your back with knees bent, then flatten your lower back against the floor by tightening your abs and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises are gentle and can help alleviate lower back discomfort while also preventing future issues. However, it’s always best to consult with a healthcare professional before starting any new exercise routine, especially if you have significant or chronic pain.'

In [ ]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep is essential for overall health because it supports physical repair, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7-9 hours for adults—also helps maintain a healthy immune system, emotional stability, and proper functioning of various bodily systems. Poor sleep or sleep disorders like insomnia can negatively impact health, increasing the risk of various medical conditions.'

In [ ]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing, doing progressive muscle relaxation, grounding techniques, taking short walks in nature, and listening to calming music. For headaches specifically, drinking water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gentle massage of the temples and neck, using peppermint or lavender essential oils, and maintaining a regular sleep schedule can also help.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [ ]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [ ]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and raise your shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises are recommended to alleviate discomfort and 

In [ ]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a profound impact on overall health. It is essential for physical recovery, as it allows the body to repair tissues and regenerate. Sleep also plays a critical role in mental well-being, including memory consolidation and emotional regulation. Adequate sleep supports various bodily functions, such as hormone regulation related to growth and appetite, and helps maintain a healthy immune system. Poor sleep or sleep disorders like insomnia can negatively affect physical health, cognitive performance, and immune function. Therefore, quality sleep—typically 7 to 9 hours per night—is vital for maintaining good health and overall well-being.'

In [ ]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises (e.g., inhale for 4 counts, hold, exhale)\n- Progressive muscle relaxation, tensing and releasing muscle groups\n- Grounding techniques, such as naming things you see, hear, feel, smell, and taste\n- Taking short walks, preferably in nature\n- Listening to calming music\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils like peppermint or lavender\n- Maintaining a regular sleep schedule\n- Staying hydrated by drinking plenty of water\n\nThese approaches can help alleviate stress and headaches naturally.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

When a user asks a question, there's usually only one way they phrase it, but the relevant documents might use completely different wording. By generating multiple reformulations of the query, we cast a wider net during retrieval. Each rephrased version might match against different chunks in the vector store that the original phrasing would miss. For example, if a user asks "exercises for back pain," the LLM might also generate "physical therapy for lumbar issues" and "stretches to relieve spinal discomfort" — each of these could surface different relevant documents. Then we take the union of all unique documents retrieved across every reformulation, which directly boosts recall. The tradeoff is that it adds an extra LLM call and increases retrieval time, but for knowledge bases where terminology varies a lot, the improvement in coverage is well worth it.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [ ]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [ ]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [ ]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [ ]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [ ]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'To help with lower back pain, the following exercises are recommended:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back upward (cat) and letting it sag downward (cow). Perform 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged, hold for 5 seconds, then switch sides. Do 10 repetitions on each side.\n- **Partial Crunches:** Lie on your back with knees bent, cross your arms over your chest, tighten your stomach muscles, and lift your shoulders off the floor. Perform 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat on the floor. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis upward. Hold for 10 seconds and repeat 8-12 times.\n\nAlways consult with a healthcare pr

In [ ]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly affects overall health by supporting various bodily functions. Adequate sleep (7-9 hours per night for adults) allows the body to repair tissues, strengthen the immune system, and regulate hormones related to growth and appetite. During sleep, especially in deep stages, the body undergoes essential repair and regenerative processes. Additionally, sleep is crucial for mental well-being and cognitive functions, such as memory consolidation, learning, and emotional regulation. Poor sleep or sleep disturbances like insomnia can impair these functions and increase the risk of health issues, highlighting the importance of good sleep hygiene and practices to maintain overall health.'

In [ ]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing exercises, engaging in mindfulness or meditation, doing gentle yoga or stretching, taking a warm bath, listening to calming music, and applying essential oils such as peppermint or lavender. Additionally, maintaining adequate hydration, getting regular sleep, and managing stress through hobbies and relaxation techniques can help alleviate headaches and reduce overall stress.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [ ]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [ ]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [ ]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, tighten your abs and tilt pelvis up slightly to flatten your back against the floor. Hold for 10 seconds, repeat 8-12 times.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese gentle exercises can help alleviate discomfort and prevent future episodes of lower back pain.'

In [ ]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a profound effect on overall health. It is essential for physical recovery, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and regulates hormones that influence growth and appetite. Adequate sleep, typically 7-9 hours per night for adults, supports immune function, helps manage stress, and promotes emotional stability. Poor or insufficient sleep can lead to issues such as fatigue, immune suppression, cognitive impairments, mood disturbances, and increased risk for chronic health conditions. Therefore, maintaining good sleep hygiene and creating a sleep-friendly environment are crucial for overall health and wellness.'

In [ ]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises\n- Progressive muscle relaxation\n- Grounding techniques (e.g., naming things you see, hear, feel, smell, taste)\n- Taking short walks in nature\n- Listening to calming music\n- Staying well-hydrated by drinking water\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils like peppermint or lavender\n- Maintaining a regular sleep schedule\n- Practicing mindfulness and meditation regularly\n\nThese approaches can help manage stress and alleviate headache symptoms naturally.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [ ]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [ ]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [ ]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [ ]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [ ]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help alleviate lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help relieve lower back discomfort and may prevent future episodes.'

In [ ]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical, mental, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Getting adequate sleep—typically 7 to 9 hours per night for adults—is essential for maintaining immune function, mental well-being, and physical health. Good sleep quality is influenced by sleep hygiene practices such as maintaining a consistent schedule, creating a relaxing bedtime routine, and optimizing the sleep environment. Poor sleep or sleep deprivation can lead to issues like fatigue, decreased concentration, weakened immunity, mood disturbances, and increased risk for various health conditions.'

In [ ]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises: Inhale for 4 counts, hold for 4, exhale for 4 to promote relaxation.\n- Progressive muscle relaxation: Tense and release muscle groups from toes to head.\n- Grounding techniques: Name 5 things you see, 4 you hear, 3 you feel, 2 you smell, and 1 you taste to shift focus and reduce stress.\n- Taking short walks, especially in nature, to calm the mind.\n- Listening to calming music.\n- Herbal teas such as chamomile or valerian root can help promote relaxation.\n- Applying peppermint or lavender essential oils for headache relief.\n- Staying hydrated by drinking water, as dehydration can trigger headaches.\n- Resting in a dark, quiet room during headaches.\n- Gentle massage of temples and neck muscles.\n\nRemember to consult with a healthcare professional if headaches or stress persist or worsen.'

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

in the example of FAQ, every page feels the same. You can't tell where to cut because nothing ever really changes. So you either never cut.
Lowerin the threshhold so small differences cause a brea.

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [ ]:
### YOUR CODE HERE

from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Use a stronger model for generation (not nano)
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)

# Generate from your raw docs
# NOTE: raw_docs is your unsplit document from the notebook
testset = generator.generate_with_langchain_docs(
    raw_docs, 
    testset_size=30  # 30 is enough for directional comparison
)

test_df = testset.to_pandas()
print(test_df.head())
print(f"Generated {len(test_df)} test samples")

C:\Users\medha\AppData\Local\Temp\ipykernel_27540\3197168010.py:9: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
C:\Users\medha\AppData\Local\Temp\ipykernel_27540\3197168010.py:10: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


ImportError: rapidfuzz is required for string distance. Please install it using `pip install rapidfuzz`